In [ ]:
#Structured output.

"""
Models can be requested to provide their response in a format matching a given schema.
This is useful for ensuring the output can be easily parsed and used in subsequent processing.
Langchain supports multiple schema types and methods for enforcing the structured output.

#Pydantic.
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.
"""

In [6]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.3-70b-versatile")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016E8B2F3310>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016E8B5C0510>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(discription="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description = "The director of the movie")
    rating:float = Field(description = "The movies rating out of 10")

C:\Users\anipi\AppData\Local\Temp\ipykernel_17088\3700598318.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'discription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  title:str = Field(discription="The title of the movie")


In [8]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016E8B2F3310>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016E8B5C0510>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': 

In [9]:
response = model_with_structure.invoke("Provide details about the movie inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

In [10]:
from jedi.api.classes import BaseName
#Message output.
class Movie(BaseModel):
    """A movie with details"""
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description="The year the movie was released")
    director:str = Field(..., description="The director of the move")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw = True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'j7brxzrrq', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 290, 'total_tokens': 322, 'completion_time': 0.059628568, 'completion_tokens_details': None, 'prompt_time': 0.088258181, 'prompt_tokens_details': None, 'queue_time': 0.054763397, 'total_time': 0.147886749}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb8d1-c10b-75b2-9872-356341304dd0-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'j7brxzrrq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 32,

In [13]:
#Nested Structure.

from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role : str

class MovieDetails(BaseModel):
    title: str
    year : str
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], genres=['Action', 'Sci-Fi'], budget=160.0)

In [ ]:
#TypeDict.

"""
TypeDict provides a simpler alternative using Python's built in typing, ideal whrn you don't need run time validation.

"""

In [15]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title : Annotated[str, ..., "The title of the movie"]
    year : Annotated[int, ..., "The year the movie was released"]
    director : Annotated[str, ..., "The director of the movie"]
    rating : Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avengers', 'year': 2012}

In [17]:

class Actor(TypedDict):
    name : str
    role : str

class MovieDetails(TypedDict):
    title: str
    year : int
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

In [18]:
model.profile

{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

In [ ]:
#Data Classes.

"""
A data class is a class typically containing mainly data, although there aren't really any restrictions.
You create it using the @dataclass decorator.
"""

In [20]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name : str = Field(description="The name of the person")
    email : str = Field(description="The email addess of the person")
    phone : str = Field(description="The phone number of the person")

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    response_format= ContactInfo #Auto - selects Provider Startegy.

)

result = agent.invoke(
    {
        "messages": [{
    "role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            
    }]
    }
)

print(result["structured_response"])


name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [21]:
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str
    email: str
    phone: str

model = init_chat_model("groq:llama-3.3-70b-versatile")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}
